# Week 81, Day 4: Databases at scale

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w081d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import zlib

keys = [f"ticket:{i}" for i in range(1000)]
before = {k: zlib.crc32(k.encode()) % 3 for k in keys}
after = {k: zlib.crc32(k.encode()) % 4 for k in keys}
moved = sum(before[k] != after[k] for k in keys)
print(f"adding a fourth server moved {moved / len(keys):.0%} of the keys")

In [ ]:
leader_log = []
followers = {"replica-1": 0, "replica-2": 0}      # how far each has applied

def write(change):
    leader_log.append(change)
    return len(leader_log)                       # the write's log position

def catch_up(name, steps):
    followers[name] = min(len(leader_log), followers[name] + steps)

position = write("ticket 7: status=resolved")
catch_up("replica-1", 1)
for name, applied in followers.items():
    print(name, "has the change" if applied >= position else "is behind: the user would see the old status")

In [ ]:
import heapq

shards = {
    "shard-a": [(1, "Printer"), (4, "Wifi"), (9, "Mouse")],
    "shard-b": [(2, "Login"), (3, "VPN"), (12, "Email")],
    "shard-c": [(5, "Laptop"), (6, "Phone")],
}
limit = 4
per_shard = [rows[:limit] for rows in shards.values()]      # each shard's own first 4, by id
print(list(heapq.merge(*per_shard))[:limit])

> **Key idea.** Leader-follower replication sends writes to the leader's log and copies it to followers, which serve reads. Asynchronous copying means replication lag; give users read-your-writes by routing their reads to a replica that has reached their last write's log position. On failover promote the most up-to-date follower; synchronous replication to one follower stops confirmed writes being lost. Partition by range (efficient ranges, hot spots) or hash (even load, scatter for ranges); choose a key that keeps a request's data together; cross-shard queries scatter to every shard, take each shard's top results, and merge. Consistent hashing puts servers (with many virtual nodes) and keys on a ring, so adding a server moves about 1/N of the keys instead of nearly all. Quorums with R + W > N make reads overlap writes; under a network partition, systems choose consistency or availability (CAP), often per operation.

### Exercise: Replication lag and read-your-writes

Write a class `ReplicatedStore(replica_names)`. The leader keeps a log of `(key, value)` writes and its own data; each replica has its own data and the log position it has applied (starting at 0). `write(key, value, session=None)` appends to the log, applies it on the leader, records the new log position as that session's last write (if a session is given), and returns the position (the log's length). `replicate(name, steps=None)` applies up to `steps` more log entries (all remaining if `None`) to that replica. `read(key, replica, session=None)` returns the value from that replica, unless the session has written something the replica hasn't applied yet, in which case it reads from the leader instead; it returns `(value, served_by)`, where `served_by` is the replica's name or `"leader"`. `lag()` returns a dict of replica name to how many log entries it's behind.

In [ ]:
class ReplicatedStore:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_replication():\n    \"lag, and reading your own writes\"\n    s = ReplicatedStore([\"r1\", \"r2\"])\n    assert s.write(\"ticket:7\", \"open\", session=\"ada\") == 1\n    assert s.read(\"ticket:7\", \"r1\") == (None, \"r1\")\n    assert s.read(\"ticket:7\", \"r1\", session=\"ada\") == (\"open\", \"leader\")\n    s.replicate(\"r1\")\n    assert s.read(\"ticket:7\", \"r1\", session=\"ada\") == (\"open\", \"r1\")\n    s.write(\"ticket:7\", \"resolved\", session=\"ada\")\n    s.write(\"ticket:8\", \"open\", session=\"bob\")\n    assert s.lag() == {\"r1\": 2, \"r2\": 3}\n    s.replicate(\"r2\", steps=2)\n    assert s.read(\"ticket:7\", \"r2\", session=\"ada\") == (\"resolved\", \"r2\")\n    assert s.read(\"ticket:8\", \"r2\", session=\"bob\") == (\"open\", \"leader\") and s.read(\"ticket:8\", \"r2\") == (None, \"r2\")\n    assert s.read(\"ticket:7\", \"r1\") == (\"open\", \"r1\") and s.lag() == {\"r1\": 2, \"r2\": 1}")

### Exercise: A consistent hash ring

Write a class `HashRing(nodes=(), vnodes=100)`. Each node is placed on the ring `vnodes` times, at the positions `zlib.crc32(f"{node}#{i}".encode())` for `i` from 0. `add(node)` and `remove(node)` add or remove all of a node's positions. `node_for(key)` returns the node owning the first ring position at or after `zlib.crc32(key.encode())`, wrapping round to the lowest position if there's none after it; with no nodes, raise `LookupError("the ring is empty")`. Keep the positions in a sorted list and use `bisect`. Then `moved(ring_before, ring_after, keys)`: the fraction of `keys` whose node differs between two rings, rounded to 2 places.

In [ ]:
import bisect
import copy
import zlib

class HashRing:
    ___

def moved(ring_before, ring_after, keys):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_ring():\n    \"only a fair share moves\"\n    import copy, zlib\n    keys = [f\"ticket:{i}\" for i in range(4000)]\n    ring = HashRing([\"db-1\", \"db-2\", \"db-3\"])\n    owners = {k: ring.node_for(k) for k in keys}\n    assert set(owners.values()) == {\"db-1\", \"db-2\", \"db-3\"}\n    counts = {n: list(owners.values()).count(n) for n in (\"db-1\", \"db-2\", \"db-3\")}\n    assert all(800 < c < 1900 for c in counts.values()), counts\n    bigger = copy.deepcopy(ring)\n    bigger.add(\"db-4\")\n    share = moved(ring, bigger, keys)\n    assert 0.1 < share < 0.4, share\n    assert all(bigger.node_for(k) in (owners[k], \"db-4\") for k in keys), \"keys only move to the new node\"\n    smaller = copy.deepcopy(ring)\n    smaller.remove(\"db-2\")\n    assert all(smaller.node_for(k) == owners[k] for k in keys if owners[k] != \"db-2\")\n    assert moved(ring, ring, keys) == 0.0\n\ndef test_wrap_and_empty():\n    \"the ring wraps; an empty ring refuses\"\n    import zlib\n    ring = HashRing([\"only\"], vnodes=1)\n    assert ring.node_for(\"anything\") == \"only\" and ring.node_for(\"\") == \"only\"\n    try:\n        HashRing().node_for(\"x\")\n    except LookupError as e:\n        assert str(e) == \"the ring is empty\"\n    else:\n        raise AssertionError(\"empty ring\")")

### Exercise: Range partitions and hot spots

Write a class `RangePartitions(bounds, names)`: `bounds` is a sorted list of upper bounds (exclusive) and `names` has one more name than bounds, so with bounds `["g", "n"]` and names `["p1", "p2", "p3"]`, keys below `"g"` go to `p1`, from `"g"` to below `"n"` to `p2`, and the rest to `p3`. `partition_for(key)` uses `bisect.bisect_right`. `record(key)` counts a write to that key's partition, and `hot(threshold)` returns the sorted names of partitions with more than `threshold` of all recorded writes (a fraction). `split(name, at, new_name)` splits a partition at a new bound: keys below `at` stay in `name`, the rest move to `new_name` (inserted just after it); the split point must lie inside the partition's range, else raise `ValueError(f"{at!r} is not inside {name}")`. Counts are reset after a split.

In [ ]:
import bisect

class RangePartitions:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_ranges():\n    \"date keys pile up on the newest partition\"\n    p = RangePartitions([\"2026-09\", \"2026-11\"], [\"old\", \"autumn\", \"current\"])\n    assert p.partition_for(\"2026-08-30\") == \"old\" and p.partition_for(\"2026-09\") == \"autumn\" and p.partition_for(\"2026-11-10\") == \"current\"\n    for day in [\"2026-11-01\"] * 70 + [\"2026-10-02\"] * 20 + [\"2026-05-05\"] * 10:\n        p.record(day)\n    assert p.hot(0.5) == [\"current\"] and p.hot(0.15) == [\"autumn\", \"current\"]\n    p.split(\"current\", \"2026-11-08\", \"newest\")\n    assert p.names == [\"old\", \"autumn\", \"current\", \"newest\"] and p.bounds == [\"2026-09\", \"2026-11\", \"2026-11-08\"]\n    assert p.partition_for(\"2026-11-10\") == \"newest\" and p.partition_for(\"2026-11-02\") == \"current\" and p.hot(0.0) == []\n    for bad in (\"2026-10\", \"2026-01\"):\n        try:\n            p.split(\"current\", bad, \"x\")\n        except ValueError as e:\n            assert str(e) == f\"{bad!r} is not inside current\"\n        else:\n            raise AssertionError(bad)")

### Exercise: Scatter and gather

Tickets are sharded by customer. `shards` is a dict from shard name to a list of ticket dicts (`id`, `customer_id`, `status`, `created_at`). Write `shard_for(customer_id, names)`: `names[customer_id % len(names)]` (simple modulo sharding over a fixed, sorted list of shard names). `customer_tickets(shards, customer_id)`: that customer's tickets by id, asking **only** their shard (count each shard asked in a list `asked` that the function returns too: `(tickets, asked)`). And `newest_open(shards, limit)`: the `limit` newest open tickets across all shards (by `created_at` descending, then id descending), by asking every shard for its own newest `limit` and merging with `heapq.merge` (use a key and `reverse=True`); return `(tickets, asked)`.

In [ ]:
import heapq

def shard_for(customer_id, names):
    ___

def customer_tickets(shards, customer_id):
    ___

def newest_open(shards, limit):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _shards():\n    names = [\"s0\", \"s1\", \"s2\"]\n    shards = {n: [] for n in names}\n    data = [(1, 10, \"open\", \"2026-11-01\"), (2, 11, \"open\", \"2026-11-03\"), (3, 10, \"closed\", \"2026-11-04\"), (4, 12, \"open\", \"2026-11-05\"),\n            (5, 13, \"open\", \"2026-11-02\"), (6, 11, \"open\", \"2026-11-06\"), (7, 10, \"open\", \"2026-11-06\"), (8, 14, \"open\", \"2026-10-30\")]\n    for i, c, s, t in data:\n        shards[shard_for(c, names)].append({\"id\": i, \"customer_id\": c, \"status\": s, \"created_at\": t})\n    return shards\n\ndef test_scatter():\n    \"one shard for one customer, every shard for everyone\"\n    shards = _shards()\n    assert shard_for(10, [\"s0\", \"s1\", \"s2\"]) == \"s1\" and shard_for(12, [\"s0\", \"s1\", \"s2\"]) == \"s0\"\n    tickets, asked = customer_tickets(shards, 10)\n    assert [t[\"id\"] for t in tickets] == [1, 3, 7] and asked == [\"s1\"]\n    tickets, asked = newest_open(shards, 3)\n    assert [t[\"id\"] for t in tickets] == [7, 6, 4] and sorted(asked) == [\"s0\", \"s1\", \"s2\"]\n    assert [t[\"id\"] for t in newest_open(shards, 10)[0]] == [7, 6, 4, 2, 5, 1, 8]")

### Exercise: Quorums

Write `strongly_consistent(n, r, w)`: `True` if every read quorum must overlap every write quorum (`r + w > n`), with `1 <= r, w <= n` (else `ValueError("quorums must be between 1 and n")`). Then `quorum_read(replies, r)`: `replies` is a list of `(version, value)` answers from replicas in the order they arrived, or `None` for a replica that didn't answer; once `r` real answers are in, return the value with the highest version among them; if fewer than `r` answered, raise `TimeoutError("not enough replicas answered")`. And `tolerates(n, r, w)`: how many replicas can be down while both reads and writes still succeed (`n - max(r, w)`).

In [ ]:
def strongly_consistent(n, r, w):
    ___

def quorum_read(replies, r):
    ___

def tolerates(n, r, w):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_quorums():\n    \"R + W > N\"\n    assert strongly_consistent(3, 2, 2) and not strongly_consistent(3, 1, 1) and strongly_consistent(3, 1, 3) and not strongly_consistent(5, 2, 3)\n    try:\n        strongly_consistent(3, 0, 2)\n    except ValueError as e:\n        assert str(e) == \"quorums must be between 1 and n\"\n    assert quorum_read([(3, \"resolved\"), (2, \"open\"), (1, \"new\")], 2) == \"resolved\"\n    assert quorum_read([None, (2, \"open\"), (3, \"resolved\")], 2) == \"resolved\"\n    assert quorum_read([(2, \"open\"), (3, \"resolved\"), (4, \"closed\")], 2) == \"resolved\"\n    try:\n        quorum_read([None, (2, \"open\"), None], 2)\n    except TimeoutError as e:\n        assert str(e) == \"not enough replicas answered\"\n    assert tolerates(3, 2, 2) == 1 and tolerates(5, 3, 3) == 2 and tolerates(3, 1, 3) == 0")

### Exercise: A connection pool

Databases limit how many connections they accept, so app instances share a few through a **pool**. Write a class `Pool(create, size)`: `create()` makes a new connection. `acquire()` returns an idle connection if there is one (the most recently released first), else creates a new one if fewer than `size` exist, else raises `RuntimeError("pool exhausted")` and counts a wait. `release(conn)` returns a connection to the idle list (releasing one that isn't checked out raises `ValueError("not from this pool")`). `stats()` returns `{"created": C, "in_use": U, "idle": I, "exhausted": E}`.

In [ ]:
class Pool:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pool():\n    \"reuse before creating\"\n    made = []\n    def create():\n        made.append(f\"conn{len(made) + 1}\")\n        return made[-1]\n    p = Pool(create, 2)\n    a = p.acquire()\n    b = p.acquire()\n    assert (a, b) == (\"conn1\", \"conn2\")\n    try:\n        p.acquire()\n    except RuntimeError as e:\n        assert str(e) == \"pool exhausted\"\n    p.release(a)\n    assert p.acquire() == \"conn1\" and p.stats() == {\"created\": 2, \"in_use\": 2, \"idle\": 0, \"exhausted\": 1}\n    p.release(b)\n    try:\n        p.release(b)\n    except ValueError as e:\n        assert str(e) == \"not from this pool\"\n    else:\n        raise AssertionError(\"double release\")\n    assert p.stats()[\"idle\"] == 1 and len(made) == 2")

### Bug hunt: A change that vanished

After saving, the page reloads from a replica that hasn't caught up, so the user sees their old status: this prints `after saving: open`. Route the read to the leader when the replica is behind the user's last write, so it prints `after saving: resolved`.

In [ ]:
leader = {"ticket:7": "open"}
replica = {"ticket:7": "open"}
replica_position, last_write = 0, 0

def save(key, value):
    global last_write
    leader[key] = value
    last_write += 1

def load(key):
    return replica[key]

save("ticket:7", "resolved")
print("after saving:", load("ticket:7"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"read your writes\"\n    expect_output(\"after saving: resolved\")")

### Bug hunt: A top three from one shard

To find the three cheapest products across shards, this takes the first three rows of the first shard and stops, so it prints `[(3, 'cable'), (8, 'mug'), (12, 'lamp')]`. Take each shard's own cheapest three and merge them, so it prints `[(1, 'sticker'), (2, 'pen'), (3, 'cable')]`.

In [ ]:
import heapq

shards = [[(3, "cable"), (8, "mug"), (12, "lamp")], [(2, "pen"), (9, "book")], [(1, "sticker"), (15, "chair")]]
limit = 3
print(shards[0][:limit])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"scatter and gather\"\n    expect_output(\"[(1, 'sticker'), (2, 'pen'), (3, 'cable')]\")")

### Bug hunt: Promoting the wrong follower

When the leader fails, this promotes the follower with the **lowest** log position, losing the writes it never received: this prints `promoted replica-3, lost 40 writes`. Promote the follower that's furthest along, so it prints `promoted replica-2, lost 0 writes`.

In [ ]:
positions = {"replica-1": 980, "replica-2": 1000, "replica-3": 960}
best = max(positions.values())
promoted = min(positions, key=positions.get)
print(f"promoted {promoted}, lost {best - positions[promoted]} writes")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the most up-to-date follower\"\n    expect_output(\"promoted replica-2, lost 0 writes\")")

### Puzzle: Finding a key's server on the ring

Arrange a ring lookup with wrap-around. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the rough share of keys that move when a 4th server joins a consistent hash ring of 3
q = ___   # True or False: N=3, R=2, W=2 gives reads that always see the latest write
r = ___   # "range" or "hash": partitioning that makes "all of last week" read one partition
s = ___   # where every write goes in leader-follower replication
t = ___   # which follower to promote: the one with the "highest" or "lowest" log position

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p in (0.25, \"1/4\", \"a quarter\") and q is True and r == \"range\", \"p, q and r\"\n    assert s == \"leader\" and t == \"highest\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a sharded store that rebalances

Using the given `HashRing`, write a class `ShardedStore(nodes, vnodes=100)` holding key-value data spread over nodes (each node's data is a dict in `self.data[node]`). `put(key, value)`, `get(key)` (or `None`) and `delete(key)` use the ring to find the key's node. `add_node(node)` adds it to the ring and **moves** every key whose owner changed to its new node (and nothing else), returning the number of keys moved. `remove_node(node)` refuses to remove the last node (`ValueError("can't remove the last node")`), otherwise removes it from the ring and moves each of its keys to its new owner, returning the number moved. `sizes()` returns a dict from node to the number of keys it holds, sorted by node name. `check()` returns the list of keys stored on a node that isn't their owner (an empty list means the store is consistent).

In [ ]:
import bisect
import zlib

class HashRing:
    def __init__(self, nodes=(), vnodes=100):
        self.vnodes = vnodes
        self.positions = []
        self.owner = {}
        for node in nodes:
            self.add(node)

    def _points(self, node):
        return [zlib.crc32(f"{node}#{i}".encode()) for i in range(self.vnodes)]

    def add(self, node):
        for p in self._points(node):
            bisect.insort(self.positions, p)
            self.owner[p] = node

    def remove(self, node):
        for p in self._points(node):
            self.positions.remove(p)
            del self.owner[p]

    def node_for(self, key):
        if not self.positions:
            raise LookupError("the ring is empty")
        i = bisect.bisect_left(self.positions, zlib.crc32(key.encode()))
        return self.owner[self.positions[i % len(self.positions)]]

class ShardedStore:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_store():\n    \"data follows the ring\"\n    s = ShardedStore([\"db-1\", \"db-2\", \"db-3\"])\n    keys = [f\"ticket:{i}\" for i in range(3000)]\n    for k in keys:\n        s.put(k, k.upper())\n    assert sum(s.sizes().values()) == 3000 and list(s.sizes()) == [\"db-1\", \"db-2\", \"db-3\"]\n    assert s.get(\"ticket:42\") == \"TICKET:42\" and s.get(\"ticket:99999\") is None\n    before = {k: s.ring.node_for(k) for k in keys}\n    moved = s.add_node(\"db-4\")\n    assert moved == sum(1 for k in keys if s.ring.node_for(k) != before[k]) and 300 < moved < 1200\n    assert s.check() == [] and s.get(\"ticket:42\") == \"TICKET:42\" and s.sizes()[\"db-4\"] == moved\n    gone = s.sizes()[\"db-2\"]\n    assert s.remove_node(\"db-2\") == gone and \"db-2\" not in s.sizes() and s.check() == []\n    assert all(s.get(k) == k.upper() for k in keys)\n    s.delete(\"ticket:42\")\n    assert s.get(\"ticket:42\") is None and sum(s.sizes().values()) == 2999\n\ndef test_last_node():\n    \"never lose everything\"\n    s = ShardedStore([\"solo\"])\n    s.put(\"a\", 1)\n    try:\n        s.remove_node(\"solo\")\n    except ValueError as e:\n        assert str(e) == \"can't remove the last node\"\n    else:\n        raise AssertionError(\"last node\")\n    assert s.get(\"a\") == 1")

## Warm-up

Spreading keys over servers with `hash % N` looks fine until `N` changes: then almost every key's server changes, and adding one machine means moving most of the data. Today's **consistent hashing** moves only about a quarter (the share the new server should own), and that difference is why it's used by nearly every large distributed store.

## Replication

**Replication** keeps copies of the same data on several machines. The most common arrangement is **leader and followers** (also called primary and replicas):

- every **write** goes to the **leader**, which records it in its **log** (an ordered list of changes) and applies it;
- **followers** copy the leader's log and apply the same changes in the same order;
- **reads** can go to any follower, so read capacity grows with each follower (Monday's `db_read_replicas`).

Followers usually copy **asynchronously**: the leader confirms a write before followers have it, so they're a little behind. That delay is **replication lag**, normally milliseconds, but seconds or minutes under load. Lag causes a familiar bug: a user saves a change, the page reloads, the read goes to a follower that hasn't caught up, and the change seems to have vanished. The fix is **read-your-writes consistency**: remember the log position of each user's last write, and send that user's reads to a replica that has reached it (or to the leader).

**Failover.** When the leader dies, a follower is **promoted**. Pick the one that's **furthest along** (the highest log position), or acknowledged writes are lost. Writes the old leader confirmed but no follower received are lost anyway with asynchronous replication, which is why important systems use **synchronous** replication to at least one follower: the leader confirms a write only once a follower has it too, trading some write speed for durability.

## Partitioning and sharding

When the **data** (or the write load) is too big for one leader, it's split into **partitions** (called **shards** when they live on separate servers), each holding a subset of the rows, each with its own leader and followers. Everything depends on the **partition key**, the column that decides where a row lives:

- **Range partitioning**: each partition holds a range of keys (A to F, G to M...). Range queries are efficient ("every ticket from last week" reads one or two partitions), but popular ranges get **hot**: partition by date, and all of today's writes hit one server.
- **Hash partitioning**: the partition is chosen by a hash of the key. Load spreads evenly, but range queries must ask every partition.

A good key spreads load and keeps the data a request needs together. For the helpdesk, `customer_id` is a natural choice: a customer's tickets, comments and events all live on one shard, so most requests touch one server. Queries that span customers (an agent's view of all open tickets, the statistics) must **scatter** to every shard and **gather** the results: each shard returns its best results, and the router **merges** them. For a sorted, limited query, each shard returns its own top `limit` rows, and a merge of the sorted lists (`heapq.merge`) gives the overall top `limit`.

## Consistent hashing

**Consistent hashing** places both **servers** and **keys** on a ring of hash values (0 to 2^32 - 1 for `zlib.crc32`). A key belongs to the first server **clockwise** from the key's position. Add a server, and it takes over only the keys between its position and the previous server's: roughly `1 / N` of them. Remove one, and only its keys move, to the next server along.

With few servers, their positions on the ring are uneven, so some get far more keys than others. The fix is **virtual nodes**: each server is placed on the ring many times (at the hashes of `"db-1#0"`, `"db-1#1"`, ...), which averages out the gaps; more virtual nodes give an evener spread, and a bigger server can be given more.

In Python, a ring is a sorted list of positions, with `bisect` (Week 18) finding the first position after a key's hash, wrapping round to the start at the end.

## Agreeing on answers

Some stores (Cassandra, DynamoDB, Riak) have no leader: every write goes to several of the `N` replicas, and every read asks several. The arithmetic of **quorums**: if a write waits for `W` replicas to confirm and a read asks `R` replicas, then with **`R + W > N`** every read set overlaps every write set in at least one replica, so a read always sees the latest write (taking the answer with the highest version). `N = 3, W = 2, R = 2` is the classic choice: one replica can be down for reads and writes.

The deeper limit is the **CAP theorem**: when the network splits the replicas into groups that can't talk (a **partition**), a system must choose between **consistency** (refusing requests that can't be answered with the latest data) and **availability** (answering anyway, possibly with stale data). Most systems choose per operation: a balance check must be consistent; a "likes" counter can be a little behind (**eventual consistency**: replicas converge once the network heals).

### On your own computer

PostgreSQL and MySQL both do leader-follower replication (streaming replication in PostgreSQL), and connection poolers and proxies (PgBouncer, ProxySQL) route reads to replicas. Citus (for PostgreSQL) and Vitess (for MySQL) add sharding; CockroachDB and Spanner do it, with consensus, built in. Before any of that, the cheaper steps usually come first: indexes (Week 75), caches and read replicas.

## Practice

## Quiz

## Daily challenge

## Before you go

Look up how one large service you know shards its data (many have engineering blogs explaining it), and find its partition key. Tomorrow: **designing a URL shortener**, the classic design interview question, from requirements and estimates to a working core.